In [48]:
from pyuvdata import UVData
import numpy as np
import json
import xcorr_cpu_tb as xc
import importlib

In [44]:
working_directory = '/project/s/sievers/thomasb/mars_data_23'
bits = 1
baseline_idx = 1
day = '07_11'
config_file_name = f'config_{day}.json'

In [51]:
importlib.reload(xc)
pols, rowcounts, channels, info = xc.run_script('config3.json')

ALB3 ALB1 {'name': 'Antenna 1', 'path': '/project/s/sievers/albatros/mars/202307/baseband/stn_1_central', 'coordinates': [10.123, -20.456, 30.789], 'clock_offset': 0}
ALB3 ALB2 {'name': 'Antenna 2', 'path': '/project/s/sievers/albatros/mars/202307/baseband/stn_2_east', 'coordinates': [11.234, -21.567, 31.89], 'clock_offset': -277647}
ALB3 ALB3 {'name': 'Antenna 3', 'path': '/project/s/sievers/albatros/mars/202307/baseband/stn_3_west', 'coordinates': [12.345, -22.678, 32.901], 'clock_offset': -603518}
took 0.159 seconds to read raw data on  /project/s/sievers/albatros/mars/202307/baseband/stn_1_central/16996/1699625038.raw
took 0.210 seconds to read raw data on  /project/s/sievers/albatros/mars/202307/baseband/stn_2_east/16996/1699625013.raw
took 0.188 seconds to read raw data on  /project/s/sievers/albatros/mars/202307/baseband/stn_3_west/16996/1699624999.raw
before correction 427246 1953125
-22604
after correction 427246 1975729
before correction 427246 2807617
3412
after correction 4

In [52]:
#figure out if there are missing values for Nblts (is it always full?)
    #currently, flag array is just the given mask
#translate channels into frequency values
#

In [53]:
print(info)

{'init_t': 1699625045, 'end_t': 1699625150, 'acclen': 30000, 'chanstart': 0, 'chanend': None, 'nchunks': 213}


In [3]:
# Initialize UVData object
uv = UVData()


#define data arrays  (WARNING: assume no missing data, Nblts = Ntimes * Nbls)
uv.data_arrays = pols.data
uv.flag_array = pols.mask




#define initial parameters
uv.Ntimes, uv.Nbls, uv.Npols, uv.Nfreqs = pols.shape
uv.Nblts = uv.Ntimes * uv.Nbls
uv.Nants_data = 3  #somehow make this automatic?




#frequency stuff
assert Nfreqs == len(channels)  #safety check
uv.freq_array = np.linspace(1, 18, 18) 
uv.channel_width = np.ones((uv.Nfreqs), dtype=float)





#time array computation
acclen = info["acclen"]
nchunks = info["nchunks"]
uv.time_array = np.arange(Nblts, dtype = float)  # some Julian Date
uv.lst_array = np.arange(Nblts, dtype = float)





#code to be able to tell what two antenna are present very quickly
uv.baseline_array = np.zeros(Nblts, dtype = int)




#TBD
uv.Nspws = 1
uv.Nphase = 1
uv.Nants_telescope = 2




#these tell you which antenna are present for each blt
ant_1 = np.zeros(Nblts, dtype = int)
ant_1[0], ant_1[1] = 1, 1
uv.ant_1_array = ant_1
ant_2 = np.ones(Nblts, dtype = int)
ant_2[1], ant_2[2] = 0, 0
uv.ant_2_array = ant_2



#which polarization we are talking about. there's number conventions that correspond to certain types of pols
uv.polarization_array = np.arange(1,5, dtype = int)  # e.g., XX

#spectral window stuff
uv.spw_array = np.array([0])
uv.flex_spw_id_array = np.zeros((Nfreqs), dtype=int)

#integration time and sample sizes
uv.integration_time = 10
uv.nsample_array = nsample_array



#names?
uv.telescope_name = 'FakeTelescope'
uv.antenna_names = ['ant0', 'ant1']
uv.antenna_numbers = [0,1]
uv.antenna_positions = np.zeros((2,3))
uv.instrument = 'FakeInstrument'
uv.telescope_location = (0.0, 0.0, 0.0)
uv.history = 'History'
uv.object_name = 'Object Name'



#phase center catalogue
uv.phase_center_catalog = {
    0: {
        "cat_name": "primary_center",
        "cat_type": "sidereal",
        "cat_lon": 0.0,
        "cat_lat": 0.0,
        "cat_frame": "icrs",
    }}


uv.flex_spw_id_array = np.zeros((Nfreqs), dtype=int)
uv.phase_center_app_dec = np.zeros(uv.Nblts)
uv.phase_center_app_ra = np.zeros(uv.Nblts)
uv.phase_center_frame_pa = np.zeros(uv.Nblts)
uv.phase_center_id_array = np.zeros((uv.Nblts), dtype=int)

print(uv.phase_center_catalog)
uv.uvw_array = np.ones((Nblts, 3), dtype = float)
uv.vis_units = 'uncalib'

# Write to UVH5 file
uv.write_uvh5('example_output.uvh5', clobber=True)

{0: {'cat_name': 'primary_center', 'cat_type': 'sidereal', 'cat_lon': 0.0, 'cat_lat': 0.0, 'cat_frame': 'icrs'}}


ERFA function "utcut1" yielded 20 of "dubious year (Note 3)"
ERFA function "utctai" yielded 20 of "dubious year (Note 3)"
The lst_array is not self-consistent with the time_array and telescope location. Consider recomputing with the `set_lsts_from_time_array` method.


ValueError: Some auto-correlations have non-zero uvw_array coordinates.